# Feature engineering

Read the cleaned case-level training table from notebook 2. Add affordability ratios,
employment tenure and history date features in DuckDB, then write a Parquet table for
later feature evaluation. Keep `case_id` for joins, `date_decision` and `WEEK_NUM`
for time-based validation, and `target` as the label.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import duckdb

processed_dir = Path("dataset/processed")
if not processed_dir.is_dir():
    processed_dir = Path("../dataset/processed")

cleaned_path = processed_dir / "train_base_cleaned.parquet"
if not cleaned_path.is_file():
    raise FileNotFoundError(f"Run notebook 2 to create {cleaned_path.resolve()}")

con = duckdb.connect(config={"threads": 2, "preserve_insertion_order": False})
cleaned = con.read_parquet(str(cleaned_path))

In [6]:
schema = {
    name: str(dtype)
    for name, dtype in zip(cleaned.columns, cleaned.types)
}

set(schema.values())

{'BIGINT', 'BOOLEAN', 'DATE', 'DOUBLE', 'VARCHAR'}

## Add case-level features

- `annuity_to_reported_income` divides the monthly annuity by the applicant's reported main income.
- `credit_amount_to_reported_income` divides the loan amount or credit card limit by that income.
- `employment_tenure_days` measures the days from the applicant's employment start to `date_decision`.

A ratio is null if the income is missing, nonfinite or nonpositive, or if the amount is missing, nonfinite or negative.

Tenure is null for a missing or future start date. A same-day start gives zero days.

The transformation uses the same inputs for training and scoring.

In [7]:
def add_case_features(cases):
    return cases.project("""
        *,
        CASE
            WHEN isfinite(person_1_applicant_mainoccupationinc_384A)
                 AND person_1_applicant_mainoccupationinc_384A > 0
                 AND isfinite(static_0_annuity_780A)
                 AND static_0_annuity_780A >= 0
            THEN CAST(static_0_annuity_780A AS DOUBLE)
                 / person_1_applicant_mainoccupationinc_384A
        END AS annuity_to_reported_income,
        CASE
            WHEN isfinite(person_1_applicant_mainoccupationinc_384A)
                 AND person_1_applicant_mainoccupationinc_384A > 0
                 AND isfinite(static_0_credamount_770A)
                 AND static_0_credamount_770A >= 0
            THEN CAST(static_0_credamount_770A AS DOUBLE)
                 / person_1_applicant_mainoccupationinc_384A
        END AS credit_amount_to_reported_income,
        CASE
            WHEN person_1_applicant_empl_employedfrom_271D <= date_decision
            THEN date_diff('day', person_1_applicant_empl_employedfrom_271D, date_decision)
        END AS employment_tenure_days
    """)

## Add historical date features

Use debit card and deposit openings, previous application creation dates, and
active-contract bureau update dates.

For each source, derives the follwing features,
- date_decision - min_date
- date_decision - max_date
- max_date - min_date

Note the following,
- Each field is an integer number of days.
- A single eligible date gives a zero span.
- Cases with no eligible dates have nulls for all three fields.
-  Filter the raw records to dates strictly before `date_decision` before aggregating. This keeps earlier dates when the same case also has later records. With date-only data, the strict cutoff excludes same-day events whose order is unknown.

In [8]:
def summarize_historical_dates(cases, history, date_column, feature_prefix):
    decisions = cases.project("case_id, date_decision")
    events = history.project(f"case_id, CAST({date_column} AS DATE) AS event_date")
    days_before_decision = (
        events.join(decisions, "case_id")
        .filter("event_date < date_decision")
        .project("case_id, date_diff('day', event_date, date_decision) AS days_ago")
    )
    return days_before_decision.aggregate(f"""
        case_id,
        max(days_ago) AS {feature_prefix}_earliest_days_ago,
        min(days_ago) AS {feature_prefix}_latest_days_ago,
        max(days_ago) - min(days_ago) AS {feature_prefix}_span_days
    """, "case_id")


train_dir = processed_dir.parent / "raw" / "train"
history_date_sources = (
    ("debitcard", "openingdate_857D", "debit_card_opening"),
    ("deposit", "openingdate_313D", "deposit_opening"),
    ("applprev", "creationdate_885D", "prior_application"),
    ("credit_bureau_a", "lastupdate_1112D", "active_bureau_update"),
)

featured = add_case_features(cleaned)
for source, date_column, feature_prefix in history_date_sources:
    stem = f"train_{source}_1"
    single_path = train_dir / f"{stem}.parquet"
    paths = [single_path] if single_path.is_file() else sorted(train_dir.glob(f"{stem}_*.parquet"))
    if not paths:
        raise FileNotFoundError(f"No Parquet files found for {stem} in {train_dir.resolve()}")

    history = con.read_parquet([str(path) for path in paths])
    summary = summarize_historical_dates(cleaned, history, date_column, feature_prefix)
    featured = featured.join(summary, "case_id", how="left")

## Validate and save

Check the case grain, labels, split metadata and engineered values. Report coverage of the new
features and save the source columns and derived features together.

In [21]:
cleaned.aggregate("count(*)").fetchall()

[(1526659,)]

In [20]:
cleaned_columns = set(cleaned.columns)
engineered_columns = [name for name in featured.columns if name not in cleaned_columns]
if not engineered_columns:
    raise ValueError("No engineered columns found. Rerun the feature engineering cells before saving.")

invalid_checks = "\n            OR ".join(
    f"NOT isfinite({name}) OR {name} < 0" for name in engineered_columns
)
coverage_counts = ", ".join(f"count({name})" for name in engineered_columns)

[(expected_rows,)] = cleaned.aggregate("count(*)").fetchall()
[(row_count, unique_case_ids, invalid_metadata, invalid_features, *feature_counts)] = (
    featured.aggregate(f"""
        count(*),
        count(DISTINCT case_id),
        count(*) FILTER (
            WHERE target IS NULL OR target NOT IN (0, 1)
               OR WEEK_NUM IS NULL OR date_decision IS NULL
        ),
        count(*) FILTER (WHERE {invalid_checks}),
        {coverage_counts}
    """).fetchall()
)

assert row_count == expected_rows > 0, "Feature engineering must preserve the input rows."
assert unique_case_ids == row_count, "case_id must be unique and non-null."
assert invalid_metadata == 0, f"Found {invalid_metadata:,} rows with invalid labels or split metadata."
assert invalid_features == 0, f"Found {invalid_features:,} rows with invalid engineered values."

for name, count in zip(engineered_columns, feature_counts):
    print(f"{name}: {count:,} of {row_count:,} cases ({count / row_count:.1%})")

output_path = cleaned_path.with_name("train_base_engineered.parquet")
with TemporaryDirectory(dir=output_path.parent) as temp_dir:
    staged_path = Path(temp_dir) / output_path.name
    featured.write_parquet(str(staged_path), row_group_size=32768)
    staged_path.replace(output_path)

print(f"Saved {row_count:,} cases to {output_path.resolve()}")

annuity_to_reported_income: 1,526,650 of 1,526,659 cases (100.0%)
credit_amount_to_reported_income: 1,526,650 of 1,526,659 cases (100.0%)
employment_tenure_days: 566,385 of 1,526,659 cases (37.1%)
debit_card_opening_earliest_days_ago: 105,087 of 1,526,659 cases (6.9%)
debit_card_opening_latest_days_ago: 105,087 of 1,526,659 cases (6.9%)
debit_card_opening_span_days: 105,087 of 1,526,659 cases (6.9%)
deposit_opening_earliest_days_ago: 105,111 of 1,526,659 cases (6.9%)
deposit_opening_latest_days_ago: 105,111 of 1,526,659 cases (6.9%)
deposit_opening_span_days: 105,111 of 1,526,659 cases (6.9%)
prior_application_earliest_days_ago: 1,207,863 of 1,526,659 cases (79.1%)
prior_application_latest_days_ago: 1,207,863 of 1,526,659 cases (79.1%)
prior_application_span_days: 1,207,863 of 1,526,659 cases (79.1%)
active_bureau_update_earliest_days_ago: 559,794 of 1,526,659 cases (36.7%)
active_bureau_update_latest_days_ago: 559,794 of 1,526,659 cases (36.7%)
active_bureau_update_span_days: 559,794 